# 05 · Backtest leakage

A little bit of future information makes a useless system look brilliant. Three kinds show up in a project like this one:

1. **News leakage** — using a headline published after the forecast time.
2. **Model training data leakage** — the model already 'knows' the outcome because it happened before the model's training cutoff.
3. **Question selection leakage** — cherry-picking which questions to score after seeing how they resolved.

This codebase prevents (1) directly and avoids (2) and (3) by construction: forecasting is always forward-only, on whatever questions are currently open, never selected with hindsight.

In [ ]:
import sys

sys.path.insert(0, "../src")
from datetime import datetime, timedelta

import numpy as np

from forecaster.retrieval import ArticleLike, select_articles
from forecaster.scoring import brier_score

## Demonstrating news leakage's effect on score

Simulate a forecaster that, when leakage is allowed, gets to see a headline published *after* the question resolved — effectively cheating.

In [ ]:
rng = np.random.default_rng(0)
asof = datetime(2026, 6, 1)

# an 'oracle' headline that only exists because the outcome is already known
leaked_headline = ArticleLike(
    title="Confirmed: the outcome has already happened",
    published_at=asof + timedelta(days=5),  # published AFTER asof
    url="https://example.invalid/leak",
    domain="example.invalid",
)
legitimate_headline = ArticleLike(
    title="Analysts expect a close outcome",
    published_at=asof - timedelta(days=1),  # published BEFORE asof
    url="https://example.invalid/legit",
    domain="example.invalid",
)

with_guard = select_articles([leaked_headline, legitimate_headline], "q", asof)
print("articles visible to the forecaster:", [a.title for a in with_guard])
print(
    "the leaked, future-published headline was excluded:",
    leaked_headline.title not in [a.title for a in with_guard],
)

`select_articles` enforces `published_at < asof` unconditionally — there is no code path in this project that can see a headline from the future, regardless of how 'relevant' it looks.

## Why an inflated backtest looks so convincing

Suppose a buggy pipeline accidentally let 20% of forecasts see slightly leaked information. Even a small leak rate can make the measured Brier score look dramatically better than true forecasting skill would ever produce — which is exactly why this project treats the leakage guard as a hard invariant, not a tunable setting.

In [ ]:
n = 500
true_p = rng.uniform(0.1, 0.9, n)
outcomes = (rng.random(n) < true_p).astype(float)

honest_forecast = np.clip(true_p + rng.normal(0, 0.15, n), 0.02, 0.98)
leak_rate = 0.2
leaked_mask = rng.random(n) < leak_rate
leaky_forecast = honest_forecast.copy()
leaky_forecast[leaked_mask] = outcomes[leaked_mask] * 0.98 + 0.01  # 'knows' the answer

print("honest Brier:", round(brier_score(honest_forecast, outcomes), 4))
print("leaky Brier (20% of forecasts cheat):", round(brier_score(leaky_forecast, outcomes), 4))

**Takeaway:** if a backtest ever looks *too* good, check the leakage guard before celebrating — `select_articles`'s `published_at < asof` filter and the forward-only design of `pipeline.forecast` are the two places this project depends on to stay honest.